# Phase 0 — P0.5(a) chọn lại k chỉ trên dev, chạy lại B3b và GAT · Colab T4

Dựng lại đồ thị k ∈ {5, 10, 20} (+ cầu nối) trong **cùng không gian `X_tab`**, chọn k theo từng fold chỉ bằng nhãn 4 fold dev còn lại, rồi chạy lại `fusion/baselines.py` (B3b và các baseline) và `fusion/gnn.py` (GAT, và GAT tắt đồ thị) cho từng k. Số của báo cáo 26/09 không được dùng.

Ước tính trên T4: mỗi k khoảng 6 phút baselines + 6–9 phút mỗi lần GAT.
Cần nhánh `PAG` đã push. Không cần ảnh. Dữ liệu cần dùng nằm trong git.

In [ ]:
import torch
!git clone -q -b PAG https://github.com/Do-Trung-Quan/DO-AN-TOT-NGHIEP-E22-T02.git /content/repo
%cd /content/repo
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO GPU')
!git log -1 --format='%h %s'

In [ ]:
# Dựng đồ thị k=5,10,20 trong X_tab và chọn k chỉ bằng nhãn dev (4 fold còn lại cho từng fold)
!python -u -m phase0.p05_select_k

In [ ]:
import json, subprocess
sel = json.load(open('phase0/output/p05_k_selection.json'))
ks = sel['k_ung_vien']; chosen = sorted(set(sel['k_chon_theo_fold'].values()) | {sel['k_chon_toan_dev']})
print('k ung vien', ks, '| k duoc chon o it nhat mot fold', chosen)

def run(cmd):
    print('>>', cmd); r = subprocess.run(cmd, shell=True); assert r.returncode == 0, cmd

# control: moi k; frozen: chi cac k duoc chon
plan = [('control', k) for k in ks] + [('frozen', k) for k in chosen]
for source, k in plan:
    d = f'phase0/output/runs/{source}_k{k}'
    g = f'phase0/output/graphs/graph_ketqua_k{k}_bridge.npz'
    run(f'python -u fusion/baselines.py --source {source} --graph {g} --out-dir {d}')
    run(f'python -u fusion/gnn.py --source {source} --graph {g} --out-dir {d} --phase3-oof {d}/phase3_oof_{source}.npz')
    if source == 'control':
        run(f'python -u fusion/gnn.py --source {source} --graph {g} --out-dir {d} --phase3-oof {d}/phase3_oof_{source}.npz --no-graph')

In [ ]:
!python -u -m phase0.p05_assemble
!cat phase0/output/p05_rerun_gat_b3b.md

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/phase0_results
!cp phase0/output/p05_* /content/drive/MyDrive/phase0_results/
!cp -r phase0/output/graphs /content/drive/MyDrive/phase0_results/
!cp -r phase0/output/runs /content/drive/MyDrive/phase0_results/